# Step 1 - Generate a risk-weighted Zurich cycling network

Goal: convert a normal OSM cycling graph into a graph whose edges carry safety attributes for Step 2 route search.

Final outputs: `outputs/zurich_hourly_edge_safety_score.csv` and `outputs/zurich_bicycle_accident_heatmap.html`.

Important edge attributes created here before export: `length`, `accident_count_50m`, `accident_density`, `accident_risk_norm`, `road_type_penalty`, `safety_cost`, plus hourly-smoothed fields like `accident_count_h08`, `accident_density_h08`, `accident_risk_norm_h08`, and `safety_cost_h08`.

`safety_cost` is the routing cost: larger values make an edge less attractive for a safety-oriented route. `safety_score_norm_hXX` is a percentile rank of the hourly safety penalty, where higher values mean the edge is riskier relative to other Zurich cycling edges in that hour.

## 1. Imports and configuration

This cell defines the study area, coordinate system, accident search radius, risk weights, and output files. Keep these parameters together so the assumptions behind Step 1 are easy to audit.

In [20]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import HeatMap
import osmnx as ox

ox.settings.use_cache = True
ox.settings.log_console = True

BASE_DIR = Path.cwd()
ACCIDENT_CSV = BASE_DIR / "roadtrafficaccidentlocations.csv"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Study area and CRS. EPSG:2056 uses meters, which is required for buffer and density calculations.
PLACE_NAME = "Zurich, Switzerland"
LOCAL_CRS = "EPSG:2056"  # Swiss LV95, meters. Accident coordinates are already in this CRS.
SAFETY_RADIUS_M = 50

# These weights control how strongly the final route search avoids risk.
# Increase ACCIDENT_RISK_WEIGHT if you want stronger hotspot avoidance in Step 2.
ACCIDENT_RISK_WEIGHT = 3.0
ROAD_TYPE_WEIGHT = 1.0
# Smoothing blends each edge-hour count with the city-wide hourly accident pattern.
HOURLY_SMOOTHING_ALPHA = 2.0
MAP_SAFETY_HOUR = 8
HOURS = list(range(24))
HOUR_LABELS = [f"h{hour:02d}" for hour in HOURS]

# Step 1 output artifacts kept for submission/inspection.
EDGES_CSV = OUTPUT_DIR / "zurich_hourly_edge_safety_score.csv"
RISK_SOURCE = "zurich_bicycle_accidents_2011_2025_osm_50m_hourly_smoothed"
ACCIDENT_HEATMAP_HTML = OUTPUT_DIR / "zurich_bicycle_accident_heatmap.html"


## 2. Helper functions

These utility functions keep the later spatial analysis cells readable. They handle CSV booleans, OSM highway tags, score normalization, file-safe values, and the OSMnx 2.x edge-length API difference.

In [21]:
def to_bool(series: pd.Series) -> pd.Series:
    """Handle CSV booleans whether pandas reads them as bools or strings."""
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().map({"true": True, "false": False})


def as_list(value):
    """Normalize scalar/list-like OSM tag values into a Python list."""
    if isinstance(value, (list, tuple, set)):
        return list(value)
    if pd.isna(value):
        return []
    return [value]


ROAD_TYPE_PENALTY = {
    "cycleway": 0.00,
    "path": 0.05,
    "track": 0.05,
    "pedestrian": 0.05,
    "living_street": 0.10,
    "residential": 0.15,
    "service": 0.20,
    "unclassified": 0.25,
    "tertiary": 0.30,
    "tertiary_link": 0.35,
    "secondary": 0.45,
    "secondary_link": 0.50,
    "primary": 0.65,
    "primary_link": 0.70,
    "trunk": 0.90,
    "trunk_link": 0.90,
    "motorway": 1.00,
    "motorway_link": 1.00,
}


def road_type_penalty(highway_value) -> float:
    """Return the strictest penalty if an OSM edge has multiple highway tags."""
    highway_tags = [str(item) for item in as_list(highway_value)]
    if not highway_tags:
        return 0.30
    return max(ROAD_TYPE_PENALTY.get(tag, 0.30) for tag in highway_tags)


def minmax_norm(series: pd.Series) -> pd.Series:
    """Scale numeric values to 0-1 using min-max normalization."""
    values = series.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0)
    min_value = values.min()
    max_value = values.max()
    if np.isclose(max_value, min_value):
        return pd.Series(0.0, index=series.index)
    return (values - min_value) / (max_value - min_value)


def percentile_rank_norm(series: pd.Series) -> pd.Series:
    """Return each value's percentile rank; this is less sensitive to extreme outliers."""
    values = series.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0)
    return values.rank(method="average", pct=True)


def scalar(value):
    """Convert numpy scalars to plain Python types before graph attribute updates."""
    if isinstance(value, np.generic):
        return value.item()
    return value


def add_edge_lengths_if_needed(G):
    """OSMnx 2.x fix: use ox.distance.add_edge_lengths only if length is missing."""
    has_length = all("length" in data for _, _, _, data in G.edges(keys=True, data=True))
    if has_length:
        return G
    return ox.distance.add_edge_lengths(G)


def stringify_file_value(value):
    """Convert non-scalar values to strings for CSV output."""
    if isinstance(value, (list, tuple, set, dict)):
        return str(value)
    return value

## 3. Load Zurich accident data and keep bicycle-related accidents

The CSV uses Swiss LV95 coordinates: `AccidentLocation_CHLV95_E` and `AccidentLocation_CHLV95_N`. This cell keeps only bicycle-related accidents, validates the accident hour, and converts points into a GeoDataFrame in the local metric CRS.

In [22]:
accident_cols = [
    "AccidentUID",
    "AccidentType_en",
    "AccidentSeverityCategory_en",
    "AccidentInvolvingBicycle",
    "RoadType_en",
    "AccidentLocation_CHLV95_E",
    "AccidentLocation_CHLV95_N",
    "CantonCode",
    "MunicipalityCode",
    "AccidentYear",
    "AccidentMonth_en",
    "AccidentWeekDay_en",
    "AccidentHour",
]

# Load only the columns needed for Step 1 to keep memory use modest.
accidents_raw = pd.read_csv(ACCIDENT_CSV, usecols=accident_cols)
accidents_raw["AccidentInvolvingBicycle"] = to_bool(accidents_raw["AccidentInvolvingBicycle"])

# Keep bicycle accidents with valid coordinates and a valid 0-23 accident hour.
bike_accidents = accidents_raw.loc[accidents_raw["AccidentInvolvingBicycle"]].copy()
bike_accidents["AccidentHour"] = pd.to_numeric(bike_accidents["AccidentHour"], errors="coerce")
bike_accidents = bike_accidents.dropna(
    subset=["AccidentLocation_CHLV95_E", "AccidentLocation_CHLV95_N", "AccidentHour"]
)
bike_accidents["AccidentHour"] = bike_accidents["AccidentHour"].astype(int)
bike_accidents = bike_accidents[bike_accidents["AccidentHour"].between(0, 23)]

# Build point geometry directly in Swiss LV95 so distance operations are in meters.
bike_accidents_gdf = gpd.GeoDataFrame(
    bike_accidents,
    geometry=gpd.points_from_xy(
        bike_accidents["AccidentLocation_CHLV95_E"],
        bike_accidents["AccidentLocation_CHLV95_N"],
    ),
    crs=LOCAL_CRS,
)


In [23]:
bike_accidents_gdf[[
    "AccidentUID",
    "AccidentSeverityCategory_en",
    "RoadType_en",
    "AccidentYear",
    "AccidentHour",
    "geometry",
]].head()

,AccidentUID,AccidentSeverityCategory_en,RoadType_en,AccidentYear,AccidentHour,geometry
0,000A1673CB070226E0630AB38B0280E9,Accident with property damage,Minor road,2023,17,POINT (2682838 1247835)
6,001CD2B6B85F0194E0630AB38B02FD86,Accident with property damage,Principal road,2023,10,POINT (2683242 1252045)
7,001CD2B6C9690194E0630AB38B02FD86,Accident with light injuries,Minor road,2023,23,POINT (2680561 1248289)
20,0095A86F3AE3013AE0630AB38B02824F,Accident with severe injuries,Minor road,2023,8,POINT (2683039 1248868)
24,00A861F4DD5602ACE0630AB38B02BB7B,Accident with light injuries,Minor road,2023,12,POINT (2683397 1247381)


## 3.1 Bicycle accident heatmap

This map is an early visual check for the accident data before those points are converted into edge-level safety attributes. It also gives Step 3 a ready-made accident hotspot layer.

In [24]:
# Folium expects latitude/longitude coordinates, so reproject the accident points to WGS84.
bike_accidents_wgs84 = bike_accidents_gdf.to_crs("EPSG:4326")
bike_accidents_wgs84["lat"] = bike_accidents_wgs84.geometry.y
bike_accidents_wgs84["lon"] = bike_accidents_wgs84.geometry.x

map_center = [
    bike_accidents_wgs84["lat"].mean(),
    bike_accidents_wgs84["lon"].mean(),
]

accident_heatmap = folium.Map(
    location=map_center,
    zoom_start=12,
    tiles="cartodbpositron",
)

# HeatMap takes a simple list of [lat, lon] pairs.
heat_data = bike_accidents_wgs84[["lat", "lon"]].dropna().values.tolist()
HeatMap(
    heat_data,
    name="Bicycle accident hotspots",
    radius=14,
    blur=18,
    min_opacity=0.25,
).add_to(accident_heatmap)

folium.LayerControl(collapsed=False).add_to(accident_heatmap)
accident_heatmap.save(ACCIDENT_HEATMAP_HTML)

## 4. Download and project the OSM cycling network

`network_type="bike"` asks OSMnx for a cycling-compatible street network. The graph is projected to Swiss LV95 so that the 50 m buffer and edge lengths are in meters.

In [25]:
# Download a cycling-compatible OSM graph for the study area.
G = ox.graph_from_place(
    PLACE_NAME,
    network_type="bike",
    simplify=True,
    retain_all=False,
)
# OSMnx usually provides length, but this helper keeps the notebook compatible across versions.
G = add_edge_lengths_if_needed(G)
G_proj = ox.project_graph(G, to_crs=LOCAL_CRS)

# Convert the graph to GeoDataFrames so spatial joins can be done with GeoPandas.
nodes_proj, edges_proj = ox.graph_to_gdfs(
    G_proj,
    nodes=True,
    edges=True,
    fill_edge_geometry=True,
)

# Recalculate edge length after projection so it matches the metric geometry.
edges_proj["length"] = edges_proj.geometry.length.astype(float)

{
    "nodes": len(nodes_proj),
    "edges": len(edges_proj),
    "crs": str(edges_proj.crs),
    "total_edge_km": round(edges_proj["length"].sum() / 1000, 2),
}

{'nodes': 20227,
 'edges': 45250,
 'crs': 'EPSG:2056',
 'total_edge_km': np.float64(2755.08)}

## 5. Count bicycle accidents within 50 m of each edge

A single accident can be counted for more than one edge if multiple road segments are within the 50 m radius. This cell creates both total counts and hourly counts (`accident_count_h00` ... `accident_count_h23`) for each edge.

In [26]:
# Buffer each edge by 50 m, then count bicycle accidents whose points fall inside that buffer.
edge_buffers = edges_proj.reset_index()[["u", "v", "key", "geometry"]].copy()
edge_buffers["geometry"] = edge_buffers.geometry.buffer(SAFETY_RADIUS_M)
edge_buffers = gpd.GeoDataFrame(edge_buffers, geometry="geometry", crs=LOCAL_CRS)

# The spatial join links each accident point to every nearby edge buffer.
joined = gpd.sjoin(
    bike_accidents_gdf[["AccidentUID", "AccidentHour", "geometry"]],
    edge_buffers,
    how="left",
    predicate="within",
)

matched = joined.dropna(subset=["u", "v", "key"]).copy()
matched[["u", "v", "key"]] = matched[["u", "v", "key"]].astype(int)

# Total accident count per edge across all years and hours.
edge_counts = (
    matched
    .groupby(["u", "v", "key"])
    .size()
    .rename("accident_count_50m")
)

edges_proj["accident_count_50m"] = (
    edge_counts.reindex(edges_proj.index).fillna(0).astype(int)
)

# Hourly accident counts per edge. Missing hours are filled with zero below.
hourly_counts = (
    matched
    .groupby(["u", "v", "key", "AccidentHour"])
    .size()
    .unstack("AccidentHour", fill_value=0)
)

for hour in HOURS:
    col = f"accident_count_h{hour:02d}"
    if hour in hourly_counts.columns:
        edges_proj[col] = hourly_counts[hour].reindex(edges_proj.index).fillna(0).astype(int)
    else:
        edges_proj[col] = 0

hourly_count_cols = [f"accident_count_h{hour:02d}" for hour in HOURS]
edges_proj[["accident_count_50m", *hourly_count_cols]].describe()

,accident_count_50m,accident_count_h00,accident_count_h01,accident_count_h02,accident_count_h03,accident_count_h04,accident_count_h05,accident_count_h06,accident_count_h07,accident_count_h08,...,accident_count_h14,accident_count_h15,accident_count_h16,accident_count_h17,accident_count_h18,accident_count_h19,accident_count_h20,accident_count_h21,accident_count_h22,accident_count_h23
count,45250.000000,45250.000000,45250.000000,45250.000000,45250.000000,45250.000000,45250.000000,45250.000000,45250.000000,45250.000000,...,45250.000000,45250.000000,45250.000000,45250.000000,45250.000000,45250.000000,45250.000000,45250.000000,45250.000000,45250.000000
mean,3.301105,0.052641,0.053260,0.030033,0.025481,0.022829,0.019470,0.065149,0.192022,0.277768,...,0.186298,0.186232,0.248884,0.316287,0.281503,0.190696,0.109812,0.087094,0.084862,0.069481
std,6.096668,0.253797,0.282441,0.193040,0.168690,0.162810,0.149391,0.296276,0.574425,0.734355,...,0.530356,0.542389,0.635827,0.789376,0.721489,0.545515,0.394142,0.349898,0.337958,0.323300
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,163.000000,6.000000,8.000000,6.000000,3.000000,3.000000,3.000000,7.000000,7.000000,12.000000,...,8.000000,8.000000,14.000000,13.000000,13.000000,8.000000,6.000000,5.000000,6.000000,11.000000


## 6. Compute total and hourly-smoothed safety costs

`accident_density` is accidents per kilometer of edge length. The hourly fields use smoothing so that sparse hourly data does not create unstable zero-risk or spike-risk edges.

The core routing formula is `safety_cost = length * (1 + accident risk term + road type term)`. The exported hourly `safety_score` is derived from this cost via the safety penalty percentile, so it stays comparable across edges without being dominated by a few extreme outliers.

In [27]:
# Accident density makes long and short edges comparable.
length_km = (edges_proj["length"] / 1000).replace(0, np.nan)
edges_proj["accident_density"] = (
    edges_proj["accident_count_50m"] / length_km
).replace([np.inf, -np.inf], np.nan).fillna(0)

# Accident density is normalized to 0-1 before entering the routing cost formula.
edges_proj["accident_risk_norm"] = minmax_norm(edges_proj["accident_density"])
# Road type penalty adds a baseline cycling-safety cost from OSM highway class.
edges_proj["road_type_penalty"] = edges_proj.get("highway", pd.Series(index=edges_proj.index)).apply(
    road_type_penalty
)

# safety_cost is the value Step 2 can use as the NetworkX/OSMnx path-search weight.
edges_proj["safety_cost"] = edges_proj["length"] * (
    1
    + ACCIDENT_RISK_WEIGHT * edges_proj["accident_risk_norm"]
    + ROAD_TYPE_WEIGHT * edges_proj["road_type_penalty"]
)
# safety_penalty removes the pure distance effect and keeps only the added risk multiplier.
edges_proj["safety_penalty"] = (edges_proj["safety_cost"] / edges_proj["length"]) - 1
# Percentile rank avoids a few extreme edges compressing most scores near zero.
edges_proj["safety_score_norm"] = percentile_rank_norm(edges_proj["safety_penalty"])

# City-wide hourly accident distribution, used as a prior for sparse edge-hour counts.
global_hour_share = (
    bike_accidents_gdf["AccidentHour"]
    .value_counts(normalize=True)
    .reindex(HOURS, fill_value=0)
    .sort_index()
)

hourly_count_cols = []
hourly_density_cols = []
hourly_risk_norm_cols = []
hourly_safety_cost_cols = []
hourly_safety_penalty_cols = []
hourly_safety_score_cols = []

# Repeat the same safety-cost calculation for each hour of the day.
for hour in HOURS:
    label = f"h{hour:02d}"
    count_col = f"accident_count_{label}"
    density_col = f"accident_density_{label}"
    risk_col = f"accident_risk_norm_{label}"
    cost_col = f"safety_cost_{label}"
    penalty_col = f"safety_penalty_{label}"
    score_col = f"safety_score_norm_{label}"

    # Blend the observed edge-hour count with the city-wide hourly pattern.
    expected_hour_count = edges_proj["accident_count_50m"] * global_hour_share.loc[hour]
    smoothed_hour_count = (
        edges_proj[count_col] + HOURLY_SMOOTHING_ALPHA * expected_hour_count
    ) / (1 + HOURLY_SMOOTHING_ALPHA)

    edges_proj[f"accident_count_smoothed_{label}"] = smoothed_hour_count
    edges_proj[density_col] = (smoothed_hour_count / length_km).replace(
        [np.inf, -np.inf], np.nan
    ).fillna(0)
    # Build hourly normalized risk and hourly safety routing cost.
    edges_proj[risk_col] = minmax_norm(edges_proj[density_col])
    edges_proj[cost_col] = edges_proj["length"] * (
        1
        + ACCIDENT_RISK_WEIGHT * edges_proj[risk_col]
        + ROAD_TYPE_WEIGHT * edges_proj["road_type_penalty"]
    )
    # The exported hourly safety score is a relative rank of the hourly penalty.
    edges_proj[penalty_col] = (edges_proj[cost_col] / edges_proj["length"]) - 1
    edges_proj[score_col] = percentile_rank_norm(edges_proj[penalty_col])

    hourly_count_cols.extend([count_col, f"accident_count_smoothed_{label}"])
    hourly_density_cols.append(density_col)
    hourly_risk_norm_cols.append(risk_col)
    hourly_safety_cost_cols.append(cost_col)
    hourly_safety_penalty_cols.append(penalty_col)
    hourly_safety_score_cols.append(score_col)

hourly_attr_cols = (
    hourly_count_cols
    + hourly_density_cols
    + hourly_risk_norm_cols
    + hourly_safety_penalty_cols
    + hourly_safety_score_cols
    + hourly_safety_cost_cols
)

risk_cols = [
    "length",
    "highway",
    "accident_count_50m",
    "accident_density",
    "accident_risk_norm",
    "road_type_penalty",
    "safety_penalty",
    "safety_score_norm",
    "safety_cost",
]

preview_hour = f"h{MAP_SAFETY_HOUR:02d}"
preview_cols = risk_cols + [
    f"accident_count_{preview_hour}",
    f"accident_count_smoothed_{preview_hour}",
    f"accident_density_{preview_hour}",
    f"accident_risk_norm_{preview_hour}",
    f"safety_score_norm_{preview_hour}",
    f"safety_cost_{preview_hour}",
]


c:\ProgramData\anaconda3\Lib\site-packages\geopandas\geodataframe.py:1968: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
c:\ProgramData\anaconda3\Lib\site-packages\geopandas\geodataframe.py:1968: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
c:\ProgramData\anaconda3\Lib\site-packages\geopandas\geodataframe.py:1968: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consid

## 6.1 Hourly edge safety map

Green edges are relatively safer; red edges have higher relative safety penalty from nearby bicycle accidents and road type. The color uses the selected hourly `safety_score_norm_hXX` as a percentile rank, not raw `safety_cost`, so long roads are not automatically colored as more dangerous.

In [28]:
def safety_color(score):
    """Map a 0-1 relative safety score to a green-yellow-red color ramp."""
    if score < 0.20:
        return "#1a9850"  # safer green
    if score < 0.40:
        return "#91cf60"
    if score < 0.60:
        return "#fee08b"
    if score < 0.80:
        return "#fc8d59"
    return "#d73027"  # higher-risk red


# Choose which hourly safety score to visualize on the map.
map_hour_label = f"h{MAP_SAFETY_HOUR:02d}"
map_count_col = f"accident_count_{map_hour_label}"
map_smoothed_count_col = f"accident_count_smoothed_{map_hour_label}"
map_density_col = f"accident_density_{map_hour_label}"
map_score_col = f"safety_score_norm_{map_hour_label}"
map_cost_col = f"safety_cost_{map_hour_label}"

map_cols = [
    "u",
    "v",
    "key",
    "highway",
    "length",
    "accident_count_50m",
    "accident_density",
    "accident_risk_norm",
    "road_type_penalty",
    "safety_penalty",
    "safety_score_norm",
    "safety_cost",
    map_count_col,
    map_smoothed_count_col,
    map_density_col,
    map_score_col,
    map_cost_col,
    "geometry",
]

# Folium needs WGS84 geometries; keep only the fields needed in the tooltip.
edges_safety_map = edges_proj.reset_index()[map_cols].copy().to_crs("EPSG:4326")
edges_safety_map["highway"] = edges_safety_map["highway"].apply(stringify_file_value)

for col in [
    "length",
    "accident_density",
    "accident_risk_norm",
    "road_type_penalty",
    "safety_penalty",
    "safety_score_norm",
    "safety_cost",
    map_smoothed_count_col,
    map_density_col,
    map_score_col,
    map_cost_col,
]:
    edges_safety_map[col] = edges_safety_map[col].astype(float).round(4)

minx, miny, maxx, maxy = edges_safety_map.total_bounds
edge_safety_map = folium.Map(
    location=[(miny + maxy) / 2, (minx + maxx) / 2],
    zoom_start=12,
    tiles="cartodbpositron",
)

# Draw every edge colored by the selected hourly safety score.
folium.GeoJson(
    edges_safety_map.to_json(),
    name=f"Cycling edge safety score {map_hour_label}",
    style_function=lambda feature: {
        "color": safety_color(feature["properties"].get(map_score_col, 0)),
        "weight": 3,
        "opacity": 0.85,
    },
    highlight_function=lambda feature: {
        "weight": 6,
        "opacity": 1.0,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "highway",
            "length",
            map_count_col,
            map_smoothed_count_col,
            map_density_col,
            "road_type_penalty",
            map_score_col,
            map_cost_col,
        ],
        aliases=[
            "Road type",
            "Length (m)",
            f"Raw accidents {map_hour_label}",
            f"Smoothed accidents {map_hour_label}",
            f"Accident density {map_hour_label}",
            "Road type penalty",
            f"Safety score 0-1 {map_hour_label}",
            f"Safety cost {map_hour_label}",
        ],
        localize=True,
        sticky=False,
    ),
).add_to(edge_safety_map)

legend_html = f"""
<div style="position: fixed; bottom: 35px; left: 35px; z-index: 9999; background: white;
            padding: 10px 12px; border: 1px solid #999; font-size: 13px;">
  <b>Edge safety hour {MAP_SAFETY_HOUR:02d}</b><br>
  <span style="color:#1a9850;">&#9632;</span> safer<br>
  <span style="color:#91cf60;">&#9632;</span> low risk<br>
  <span style="color:#fee08b;">&#9632;</span> medium<br>
  <span style="color:#fc8d59;">&#9632;</span> high<br>
  <span style="color:#d73027;">&#9632;</span> higher risk
</div>
"""
edge_safety_map.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(edge_safety_map)
edge_safety_map.fit_bounds([[miny, minx], [maxy, maxx]])

# Display only; this diagnostic map is not saved as a separate output file.
#edge_safety_map

## 7. Write the safety attributes back into the NetworkX graph

OSMnx routing works on the NetworkX graph, not directly on the GeoDataFrame. This cell copies every Step 1 edge attribute back onto the corresponding graph edge.

In [29]:
# Base attributes exist once per edge; hourly attributes add one set per hour.
base_edge_attr_cols = [
    "length",
    "accident_count_50m",
    "accident_density",
    "accident_risk_norm",
    "road_type_penalty",
    "safety_penalty",
    "safety_score_norm",
    "safety_cost",
]
edge_attr_cols = base_edge_attr_cols + hourly_attr_cols

# Copy GeoDataFrame columns back into the MultiDiGraph edge dictionaries.
for (u, v, key), row in edges_proj[edge_attr_cols].iterrows():
    G_proj[u][v][key].update({col: scalar(row[col]) for col in edge_attr_cols})

sample_u, sample_v, sample_key = next(iter(G_proj.edges(keys=True)))
{col: G_proj[sample_u][sample_v][sample_key][col] for col in edge_attr_cols}

{'length': 4.25119350781175,
 'accident_count_50m': 20.0,
 'accident_density': 4704.561192815417,
 'accident_risk_norm': 0.3190340412654217,
 'road_type_penalty': 0.3,
 'safety_penalty': 1.257102123796265,
 'safety_score_norm': 0.9987734806629834,
 'safety_cost': 9.595377895150794,
 'accident_count_h00': 0.0,
 'accident_count_smoothed_h00': 0.2208766518146159,
 'accident_count_h01': 0.0,
 'accident_count_smoothed_h01': 0.2163067900529342,
 'accident_count_h02': 0.0,
 'accident_count_smoothed_h02': 0.11729311854983053,
 'accident_count_h03': 0.0,
 'accident_count_smoothed_h03': 0.10510682051867931,
 'accident_count_h04': 1.0,
 'accident_count_smoothed_h04': 0.4201607068052858,
 'accident_count_h05': 0.0,
 'accident_count_smoothed_h05': 0.08073422445637686,
 'accident_count_h06': 0.0,
 'accident_count_smoothed_h06': 0.26657526943143306,
 'accident_count_h07': 1.0,
 'accident_count_smoothed_h07': 1.1056399710575422,
 'accident_count_h08': 0.0,
 'accident_count_smoothed_h08': 1.11961613161

## 8. Save final Step 1 CSV

The compact CSV intentionally stores one row per edge per hour with only the fields needed for hourly safety scoring. The accident heatmap is saved earlier in Section 3.1.

In [30]:
def compact_hourly_safety_rows(edges_gdf, *, hour):
    """Return compact one-edge-one-hour rows for the final hourly safety CSV."""
    label = f"h{hour:02d}"
    safety_score_col = f"safety_score_norm_{label}"
    compact = edges_gdf.reset_index()[["u", "v", "key", "highway", safety_score_col]].copy()
    compact["edge_id"] = (
        compact["u"].astype(str)
        + "_"
        + compact["v"].astype(str)
        + "_"
        + compact["key"].astype(str)
    )
    compact["safety_score"] = compact[safety_score_col].astype(float).round(4)
    compact["time_period"] = label
    compact["highway"] = compact["highway"].apply(stringify_file_value)
    compact["risk_source"] = RISK_SOURCE
    return compact[[
        "u",
        "v",
        "key",
        "edge_id",
        "time_period",
        "safety_score",
        "highway",
        "risk_source",
    ]]


# The compact CSV has 24 rows per edge: one safety score for every hour of the day.
final_risk_scores = pd.concat(
    [compact_hourly_safety_rows(edges_proj, hour=hour) for hour in HOURS],
    ignore_index=True,
)

final_risk_scores.to_csv(EDGES_CSV, index=False)
